# Reproducability Proof - Main
This notebook showcases that this experiment is reproducible by using local llms and reproducing entire experiments word for word using the seed mechanic and temperature set to 0. 

In [1]:
# Imports
import sys, os
from pathlib import Path

# Ensure repo root on sys.path (for local package imports)
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'main.py').exists() and (p / 'hypothesis_testing').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
    return here
_REPO_ROOT = _add_repo_root_to_sys_path()

import json
import random
import shutil
import yaml
from collections import Counter
import numpy as np
from scipy.stats import chi2_contingency
from hypothesis_testing.utils_hypothesis_testing.runner import (
    list_config_files,
    select_configs,
    run_configs_in_parallel,
)


In [2]:
# Configuration paths and constants
CONFIG_DIR = _REPO_ROOT / 'hypothesis_testing' / 'reproducability_proof' / 'configs'
TERMINAL_OUTPUTS_DIR   = _REPO_ROOT / 'hypothesis_testing' / 'reproducability_proof' / 'terminal_outputs'
RESULTS_DIR= _REPO_ROOT / 'hypothesis_testing' / 'reproducability_proof' / 'results'
TRANSCRIPTS_DIR = _REPO_ROOT / 'hypothesis_testing' / 'reproducability_proof' / 'transcripts'


# Placeholder model list for participant agents

MODEL_LIST = [
    "ollama/gemma3:1b",
]

# Income class probabilities (must sum to 1.0)
# Same as in Frohlich & Oppenheimer (1992) for the initial distribution
INCOME_CLASS_PROBS = {
    'high': 0.05,
    'medium_high': 0.10,
    'medium': 0.50,
    'medium_low': 0.25,
    'low': 0.10,
}

# Ensure directories exist
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TERMINAL_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)


CONFIG_DIR, TERMINAL_OUTPUTS_DIR, RESULTS_DIR, TRANSCRIPTS_DIR


(PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/reproducability_proof/configs'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/reproducability_proof/terminal_outputs'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/reproducability_proof/results'))

## 1. Generate 3 identical configurations for reproducibility test

In [4]:
def make_agents() -> list[dict]:
    agents = []
    for i in range(0, 3):  # 3 participant agents
        agents.append({
            'name': f'Agent_{i+1}',
            'personality': 'You are an American college student',
            'model': MODEL_LIST[0],  # Use the same model for all agents (no random.choice)
            'temperature': 0, # Reproducible results
            'memory_character_limit': 2000, # set to lower value given hardware constraints of local running
            'reasoning_enabled': True,
        })
    return agents

def build_config(seed_val: int) -> dict:
    return {
        'language': 'English',
        'seed': seed_val,  # seed for reproducibility
        'agents': make_agents(),
        'utility_agent_model': "google/gemini-2.0-flash-lite-001", # cloud model to improve run time & reliability
        'utility_agent_temperature': 0.0,
        'phase2_rounds': 5,
        'distribution_range_phase2': [2, 6],
        'income_class_probabilities': INCOME_CLASS_PROBS,
        'original_values_mode': { 'enabled': True },

    }



# Generate 3 identical configs for reproducibility test
# All use the same seed (42) and temperature (0) to ensure identical results
generated_files = []
for idx in range(1, 4):  # Generate 3 identical configs
    cfg = build_config(seed_val=42)  # Same seed for all
    fname = CONFIG_DIR / f'reproducibility_condition_{idx}_config.yaml'
    with open(fname, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    generated_files.append(fname)

len(generated_files), generated_files[0] if generated_files else None

(3,
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/reproducability_proof/configs/reproducibility_condition_1_config.yaml'))

## 2. Run Configs (Parallel + Selective)

In [ ]:
# Discover all config files
configs = list_config_files(CONFIG_DIR)
print(f'Found {len(configs)} configs')

# Selection controls
SELECT_INDICES = [1, 2, 3]  # Run all 3 identical configs
SELECT_NAMES = None    # e.g., ['condition_1', 'condition_3']
CONCURRENCY = 1        # Run sequentially to avoid conflicts
TIMEOUT_SECONDS = None # e.g., 900 for 15 minutes per run

selected = select_configs(configs, include_indices=SELECT_INDICES, include_names=SELECT_NAMES)
print(f'Selected {len(selected)} configs to run')

run_results = run_configs_in_parallel(
    selected,
    concurrency=CONCURRENCY,
    logs_dir=TERMINAL_OUTPUTS_DIR,
    results_dir=RESULTS_DIR,
    timeout_sec=TIMEOUT_SECONDS,
)

# Quick summary
ok = sum(1 for r in run_results if r.get('ok'))
print(f'Completed: {ok}/{len(run_results)} OK')
run_results[:3]  # show a few

Found 3 configs
Selected 3 configs to run


## 3. Analysis — Reproducibility Check

Since all 3 runs use identical configurations (same seed, temperature 0, local Ollama model),
they should produce identical results. This section verifies reproducibility.

In [10]:
def extract_key_info(result_path: Path) -> dict:
    try:
        with open(result_path, 'r') as f:
            data = json.load(f)
        gi = data.get('general_information', {})
        return {
            'consensus_reached': gi.get('consensus_reached', False),
            'consensus_principle': gi.get('consensus_principle'),
            'final_payoffs': gi.get('final_payoffs'),
            'phase2_rounds_completed': gi.get('phase2_rounds_completed', 0)
        }
    except Exception as e:
        return {'error': str(e)}

# Load results from all 3 runs
result_files = sorted(RESULTS_DIR.glob('*_results.json'))
print(f"Found {len(result_files)} result files")

results = []
for i, rp in enumerate(result_files, 1):
    info = extract_key_info(rp)
    info['run_number'] = i
    info['filename'] = rp.name
    results.append(info)
    print(f"\nRun {i} ({rp.name}):")
    print(f"  Consensus reached: {info.get('consensus_reached')}")
    print(f"  Consensus principle: {info.get('consensus_principle')}")
    print(f"  Phase 2 rounds completed: {info.get('phase2_rounds_completed')}")
    if info.get('final_payoffs'):
        print(f"  Final payoffs: {info.get('final_payoffs')}")
    if 'error' in info:
        print(f"  Error: {info['error']}")

# Check reproducibility
if len(results) == 3:
    # Compare key outcomes
    consensus_results = [r.get('consensus_reached') for r in results]
    principle_results = [r.get('consensus_principle') for r in results]
    payoff_results = [r.get('final_payoffs') for r in results]
    
    if len(set(str(x) for x in consensus_results)) == 1 and len(set(str(x) for x in principle_results)) == 1 and len(set(str(x) for x in payoff_results)) == 1:
        print("\n✅ SUCCESS: All 3 runs produced identical results - reproducibility confirmed!")
    else:
        print("\n❌ FAILURE: Runs produced different results - reproducibility issue detected")
        print(f"Consensus results: {consensus_results}")
        print(f"Principle results: {principle_results}")
else:
    print(f"\n⚠️  Expected 3 result files, found {len(results)}")

results

Category | Count
---------|------
maximizing_floor                       | 2
maximizing_average                     | 0
maximizing_average_floor_constraint    | 8
maximizing_average_range_constraint    | 0
disagreement                           | 2


Counter({'maximizing_average_floor_constraint': 8,
         'disagreement': 2,
         'maximizing_floor': 2,
         'maximizing_average': 0,
         'maximizing_average_range_constraint': 0})